In [ ]:
import os
os.environ["ANTHROPIC_API_KEY"] = "sk-ant"
os.environ["GOOGLE_API_KEY"] = "AQ"
os.environ["HUGGINGFACEHUB_API_TOKEN"] = "hf"

# Install Libraries

In [2]:
!pip install -q youtube-transcript-api langchain-community langchain-anthropic langchain_google_genai langchain_huggingface \
               faiss-cpu tiktoken python-dotenv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 485.2/485.2 kB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 51.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.4/52.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.5/70.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 73.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 932.0/932.0 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 557.4/557.4 kB 27.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
go

In [3]:
!pip install youtube-transcript-api

In [4]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
# from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_anthropic import ChatAnthropic
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

/tmp/ipykernel_58/446290261.py:7: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


# Step 1a - Indexing (Document Ingestion)

In [5]:
video_id = "LPZh9BOjkQs" # only the ID, not full URL
try:
    # If you don't care which language, this returns the "best" one
    # transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
    ytt_api = YouTubeTranscriptApi()
    # fetched_transcript = ytt_api.fetch(video_id, languages=["en"])
    transcript_list = ytt_api.fetch(video_id, languages=["en"])

    # Flatten it to plain text
    # transcript = " ".join(chunk["text"] for chunk in transcript_list)
    # transcript = " ".join(snippet.text for snippet in fetched_transcript)
    transcript = " ".join(snippet.text for snippet in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video")

[Submit subtitle corrections at criblate.com]
Imagine you happen across a short movie script that describes a scene between a person and their AI assistant. The script has what the person asks the AI, but the AI's response has been torn off. Suppose you also have this powerful magical machine that can take any text and provide a sensible prediction of what word comes next. You could then finish the script by feeding in what you have to the machine, seeing what it would predict to start the AI's answer, and then repeating this over and over with a growing script completing the dialogue. When you interact with a chatbot, this is exactly what's happening. A large language model is a sophisticated mathematical function that predicts what word comes next for any piece of text. Instead of predicting one word with certainty, though, what it does is assign a probability to all possible next words. To build a chatbot, what you do is lay out some text that describes an interaction between a user

In [6]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='[Submit subtitle corrections at criblate.com]\nImagine you happen across a short movie script that', start=1.14, duration=2.836), FetchedTranscriptSnippet(text='describes a scene between a person and their AI assistant.', start=3.976, duration=3.164), FetchedTranscriptSnippet(text="The script has what the person asks the AI, but the AI's response has been torn off.", start=7.48, duration=5.58), FetchedTranscriptSnippet(text='Suppose you also have this powerful magical machine that can take', start=13.06, duration=3.92), FetchedTranscriptSnippet(text='any text and provide a sensible prediction of what word comes next.', start=16.98, duration=3.98), FetchedTranscriptSnippet(text='You could then finish the script by feeding in what you have to the machine,', start=21.5, duration=4.006), FetchedTranscriptSnippet(text="seeing what it would predict to start the AI's answer,", start=25.506, duration=2.862), FetchedTranscriptSnippet(te

# Step 1b - Indexing (Text Splitting)

In [7]:
video_id = "X0btK9X0Xnk" # only the ID, not full URL
try:
    # If you don't care which language, this returns the "best" one
    # transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
    ytt_api = YouTubeTranscriptApi()
    # fetched_transcript = ytt_api.fetch(video_id, languages=["en"])
    transcript_list = ytt_api.fetch(video_id, languages=["hi"])

    # Flatten it to plain text
    # transcript = " ".join(chunk["text"] for chunk in transcript_list)
    # transcript = " ".join(snippet.text for snippet in fetched_transcript)
    transcript = " ".join(snippet.text for snippet in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video")

हाय गाइस, माय नेम इज नितीश एंड यू वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना लैंड चेन प्लेलिस्ट कंटिन्यू करेंगे और अब हम फाइनली एक बहुत इंपॉर्टेंट टॉपिक कवर करने वाले हैं। और आज के वीडियो का जो टॉपिक है वो है रैग। आई एम प्रीटी श्योर आपने इस टर्म के बारे में जरूर सुना होगा। जब से ये जनरेटिव एआई की वेव आई है बहुत तरह के एप्लीकेशनेशंस बना रहे हैं लोग। बट उनमें से जो सबसे कॉमन और सबसे यूज़फुल एप्लीकेशन होता है वो शायद रैग ही होता है। तो मैंने प्लान किया है कि मैं आपको दो वीडियोस में रैग के बारे में पढ़ाऊंगा। आज का जो वीडियो है उसमें मैं मोस्टली आपको जो थ्योरिटिकल बैकग्राउंड चाहिए रैग को समझने के लिए जो कॉनसेप्चुअल बैकग्राउंड चाहिए वो समझाऊंगा और फिर नेक्स्ट वीडियो में हम लैंग चेन की हेल्प से एक रैग बेस्ड सिस्टम बनाएंगे। ठीक है? ऑन द होल आज का जो वीडियो है वो थोड़ा सा बाकी के रैग वीडियो से अलग होगा। बिकॉज़ मैं थोड़ा ज्यादा इन डेप्थ जाकर के रैग को एक्सप्लेन करने की कोशिश करूंगा। नॉट ओनली दैट मैं आपको रैग के ऊपर एक हिस्टोरिकल पर्सपेक्टिव भी दूंगा। व्हिच मीन्स कि रैग पिक्चर में आया क

In [8]:
video_id = "Gfr50f6ZBvo" # only the ID, not full URL
try:
    # If you don't care which language, this returns the "best" one
    # transcript_list = YouTubeTranscriptApi.get_transcript(video_id, languages=["en"])
    ytt_api = YouTubeTranscriptApi()
    # fetched_transcript = ytt_api.fetch(video_id, languages=["en"])
    transcript_list = ytt_api.fetch(video_id, languages=["en"])

    # Flatten it to plain text
    # transcript = " ".join(chunk["text"] for chunk in transcript_list)
    # transcript = " ".join(snippet.text for snippet in fetched_transcript)
    transcript = " ".join(snippet.text for snippet in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video")

the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to interview people until i get good enough 

In [9]:
splitter = RecursiveCharacterTextSplitter(chunk_size = 1000, chunk_overlap = 200)
# splitter = RecursiveCharacterTextSplitter(chunk_size = 1600, chunk_overlap = 200)
chunks = splitter.create_documents([transcript])

In [10]:
len(chunks)

168

In [11]:
chunks[0]

Document(metadata={}, page_content="the following is a conversation with demus hasabis ceo and co-founder of deepmind a company that has published and builds some of the most incredible artificial intelligence systems in the history of computing including alfred zero that learned all by itself to play the game of gold better than any human in the world and alpha fold two that solved protein folding both tasks considered nearly impossible for a very long time demus is widely considered to be one of the most brilliant and impactful humans in the history of artificial intelligence and science and engineering in general this was truly an honor and a pleasure for me to finally sit down with him for this conversation and i'm sure we will talk many times again in the future this is the lex friedman podcast to support it please check out our sponsors in the description and now dear friends here's demis hassabis let's start with a bit of a personal question am i an ai program you wrote to inter

In [12]:
chunks[100]
# chunks[95]

Document(metadata={}, page_content="and and kind of come up with descriptions of the electron clouds where they're gonna go how they're gonna interact when you put two elements together uh and what we try to do is learn a simulation uh uh learner functional that will describe more chemistry types of chemistry so um until now you know you can run expensive simulations but then you can only simulate very small uh molecules very simple molecules we would like to simulate large materials um and so uh today there's no way of doing that and we're building up towards uh building functionals that approximate schrodinger's equation and then allow you to describe uh what the electrons are doing and all materials sort of science and material properties are governed by the electrons and and how they interact so have a good summarization of the simulation through the functional um but one that is still close to what the actual simulation would come out with so what um how difficult is that to ask w

# Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [13]:
# embeddings =  GoogleGenerativeAIEmbeddings(model = "gemini-embedding-001")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = FAISS.from_documents(chunks, embeddings)

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [14]:
vector_store.index_to_docstore_id

{0: 'be5cd12e-2301-4fbc-b618-2f693699eb65',
 1: '247cd049-75ca-47e9-b8d2-42fde3c05a05',
 2: 'd9161f22-fe0d-4953-8e7a-c912e83d5f41',
 3: '4a86879d-8b71-4c05-923f-e6d5e2ba321d',
 4: 'ff1c4bd2-2694-4b70-8c85-647d8d6c8111',
 5: '150ca758-2b24-42d1-af23-b8f92083f0ff',
 6: '8900e4e5-b284-40c2-8539-8fbb4d5a1933',
 7: 'cadf544f-3a56-4cd1-8354-2245e2efedd1',
 8: 'a298353e-799d-41c8-aa0c-effd7a66c45f',
 9: '8503f92d-f237-4bd7-bec0-2921aa2aaca6',
 10: 'b40924dc-d6b7-424f-8bc9-b430f2857a37',
 11: '7557c81f-dfb5-4158-8368-2551e60d374a',
 12: '3f18a4ea-6c24-40e5-a279-f29943c54066',
 13: '23c2fd37-ba78-465d-b9e7-74f7a5cc39f9',
 14: '4d6858f4-bcc7-4b81-b6da-45d8e3207f94',
 15: '49af6a45-732f-47d8-ab31-021dab4536b8',
 16: '92df2d22-e706-4ecd-a0f5-be2530a9c0e7',
 17: '2f01435e-b85d-4859-b0bf-235239ad45a5',
 18: '8927cbcf-1b39-404c-88ea-0da77b284498',
 19: '6e9e6c2d-839e-4161-84c0-861aa4bcbbe1',
 20: '21eb9a65-631d-4110-8aa8-da4b4503436f',
 21: '12e327c7-020b-4b7a-a812-053eac41e08d',
 22: '4fce0577-f201-

In [15]:
vector_store.get_by_ids(['7c6f2e84-91f3-443c-9503-c290362b1ffb'])

[]

# Step 2 - Retrieval

In [16]:
retriever = vector_store.as_retriever(search_type = "similarity", search_kwargs = {"k": 4})

In [17]:
retriever

VectorStoreRetriever(tags=['FAISS', 'HuggingFaceEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x78815a4239e0>, search_kwargs={'k': 4})

In [18]:
retriever.invoke('What is deepmind')

[Document(id='b00e250e-3c2a-4762-9a67-2c544bc943ba', metadata={}, page_content="and how it works this is tough to uh ask you this question because you probably will say it's everything but let's let's try let's try to think to this because you're in a very interesting position where deepmind is the place of some of the most uh brilliant ideas in the history of ai but it's also a place of brilliant engineering so how much of solving intelligence this big goal for deepmind how much of it is science how much is engineering so how much is the algorithms how much is the data how much is the hardware compute infrastructure how much is it the software computer infrastructure yeah um what else is there how much is the human infrastructure and like just the humans interact in certain kinds of ways in all the space of all those ideas how much does maybe like philosophy how much what's the key if um uh if if you were to sort of look back like if we go forward 200 years look back what was the key 

# Step 3 - Augmentation

In [19]:
llm = ChatAnthropic(model="claude-haiku-4-5-20251001", temperature = 0.2)

In [20]:
prompt = PromptTemplate(
    template="""
        You are a helpful assistant.
        Answer ONLY from the provided transcript context.
        If the context is insufficient, just say you don't know.
        {context}
        Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [21]:
question = "is the topic of aliens discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [22]:
retrieved_docs

[Document(id='19b56d11-9353-40b0-a019-e6a47c0ec23d', metadata={}, page_content="space age we should have heard a cacophony of voices we should have joined that cacophony of voices and what we did we opened our ears and we heard nothing and many people who argue that there are aliens would say well we haven't really done exhaustive search yet and maybe we're looking in the wrong bands and and we've got the wrong devices and we wouldn't notice what an alien form was like to be so different to what we're used to but you know i'm not i don't really buy that that it shouldn't be as difficult as that like we i think we've searched enough there should be if it were everywhere if it was it should be everywhere we should see dyson's fears being put up sun's blinking in and out you know there should be a lot of evidence for those things and then there are other people argue well the sort of safari view of like well we're a primitive species still because we're not space faring yet and and and we

In [23]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"space age we should have heard a cacophony of voices we should have joined that cacophony of voices and what we did we opened our ears and we heard nothing and many people who argue that there are aliens would say well we haven't really done exhaustive search yet and maybe we're looking in the wrong bands and and we've got the wrong devices and we wouldn't notice what an alien form was like to be so different to what we're used to but you know i'm not i don't really buy that that it shouldn't be as difficult as that like we i think we've searched enough there should be if it were everywhere if it was it should be everywhere we should see dyson's fears being put up sun's blinking in and out you know there should be a lot of evidence for those things and then there are other people argue well the sort of safari view of like well we're a primitive species still because we're not space faring yet and and and we're you know there's some kind of globe like universal rule not to interfere\n\

In [24]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [25]:
final_prompt

StringPromptValue(text="\n        You are a helpful assistant.\n        Answer ONLY from the provided transcript context.\n        If the context is insufficient, just say you don't know.\n        space age we should have heard a cacophony of voices we should have joined that cacophony of voices and what we did we opened our ears and we heard nothing and many people who argue that there are aliens would say well we haven't really done exhaustive search yet and maybe we're looking in the wrong bands and and we've got the wrong devices and we wouldn't notice what an alien form was like to be so different to what we're used to but you know i'm not i don't really buy that that it shouldn't be as difficult as that like we i think we've searched enough there should be if it were everywhere if it was it should be everywhere we should see dyson's fears being put up sun's blinking in and out you know there should be a lot of evidence for those things and then there are other people argue well t

# Step 4 - Generation

In [26]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, the topic of aliens is discussed in this transcript.

**What was discussed:**

1. **The Fermi Paradox/Great Silence**: The speaker notes that despite being in the space age, we should have detected signs of alien civilizations if they were common, but instead we've "heard nothing." They argue we've done sufficient searching that we should see evidence like Dyson spheres or stars blinking in and out if aliens were prevalent.

2. **Counterarguments to alien existence**: The speaker addresses common arguments for why we haven't found aliens yet - that we haven't searched exhaustively enough, we're looking in the wrong frequencies, or we wouldn't recognize alien signals. They dismiss these as unconvincing.

3. **The "Zoo Hypothesis"**: Discussion of the idea that advanced civilizations might follow a universal rule not to interfere with primitive species like humans.

4. **Diversity of alien civilizations**: The speaker argues that if multiple alien civilizations existed, they would l

In [27]:
question = "is the topic of nuclear fusion discussed in this video? if yes then what was discussed"
retrieved_docs = retriever.invoke(question)

In [28]:
retrieved_docs

[Document(id='2ef199e0-ea2d-4764-94b0-7bba50bfffcb', metadata={}, page_content="in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be intere

In [29]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks and we look at the ones which ones are amenable to our ai methods today yes right and and and then and would be interesting from a research perspective from our point of view from an ai point of\n\

In [30]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [31]:
final_prompt

StringPromptValue(text="\n        You are a helpful assistant.\n        Answer ONLY from the provided transcript context.\n        If the context is insufficient, just say you don't know.\n        in this case in fusion we we collaborated with epfl in switzerland the swiss technical institute who are amazing they have a test reactor that they were willing to let us use which you know i double checked with the team we were going to use carefully and safely i was impressed they managed to persuade them to let us use it and um and it's a it's an amazing test reactor they have there and they try all sorts of pretty crazy experiments on it and um the the the what we tend to look at is if we go into a new domain like fusion what are all the bottleneck problems uh like thinking from first principles you know what are all the bottleneck problems that are still stopping fusion working today and then we look at we you know we get a fusion expert to tell us and then we look at those bottlenecks a

In [32]:
answer = llm.invoke(final_prompt)
print(answer.content)

Yes, nuclear fusion is discussed in this transcript.

Here's what was discussed about nuclear fusion:

1. **Collaboration with EPFL**: They collaborated with EPFL (Swiss Technical Institute) in Switzerland, which has a test reactor they were allowed to use for their research.

2. **Approach to the problem**: They identified bottleneck problems in fusion from first principles and determined which ones were amenable to their AI methods.

3. **Plasma control research**: They published a Nature paper on controlling and holding plasma in specific shapes. The work involved using AI to carve the plasma into different shapes and hold it for record amounts of time. They developed a controller capable of containing and holding plasma in structure regardless of the shape.

4. **Plasma shapes for energy production**: Different plasma shapes are better for energy production, such as droplets.

5. **Future work**: They are talking to fusion startups to identify the next problems they can tackle in t

# Building a Chain

In [33]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [34]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [35]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [36]:
parallel_chain.invoke('who is Demis')

{'context': "to get world peace because there's also other corrupting things like wanting power over people and this kind of stuff which is not necessarily satisfied by by just abundance but i think it will help um and i think uh but i think ultimately ai is not going to be run by any one person or one organization i think it should belong to the world belong to humanity um and i think maybe many there'll be many ways this will happen and ultimately um everybody should have a say in that do you have advice for uh young people in high school and college maybe um if they're interested in ai or interested in having a big impact on the world what they should do to have a career they can be proud of her to have a life they can be proud of i love giving talks to the next generation what i say to them is actually two things i i think the most important things to learn about and to find out about when you're when you're young is what are your true passions is first of all there's two things on

In [37]:
parser = StrOutputParser()

In [38]:
main_chain = parallel_chain | prompt | llm | parser

In [39]:
main_chain.invoke('Can you summarize the video')

"Based on the transcript provided, this appears to be a conversation between Lex Fridman and Demas Beard about fundamental physics and artificial intelligence. Here are the key points:\n\n**Main Topics Discussed:**\n\n1. **Physics and Fundamental Explanations** - The speakers discuss the need for deeper, simpler explanations of physics beyond the current standard model, which they acknowledge doesn't fully work.\n\n2. **Big Mysteries** - They mention that better physics explanations could help address long-standing mysteries like consciousness, life, and gravity.\n\n3. **Intelligence and Explanation** - They discuss how a sign of true intelligence is the ability to explain complex topics simply and clearly, referencing Richard Feynman as an example.\n\n4. **AI and Human Enhancement** - The conversation touches on AI systems, human-computer symbiosis through devices like phones, and the possibility of human enhancement.\n\n5. **Chess and AI History** - They discuss the history of chess 

In [40]:
# Chat with some other video_id